# Chinese Instrument Demucs — Colab Quickstart

**Isolate any Chinese instrument from music** using a Demucs model fine-tuned for single-instrument separation.

📦 **Repo:** [wushanyun64/chinese-instrument-demucs](https://github.com/wushanyun64/chinese-instrument-demucs)

This notebook clones the repo, sets up the environment, and demonstrates:
1. Installing dependencies
2. Running inference with a warm-started model
3. Separating your own audio files

## 1. Check GPU

Colab provides a free T4 GPU — verify it's available:

In [ ]:
!nvidia-smi -L

## 2. Clone the repo

In [ ]:
!git clone https://github.com/wushanyun64/chinese-instrument-demucs.git
%cd chinese-instrument-demucs

## 3. Install dependencies

Colab doesn't have `uv` by default, so we install it first:

In [ ]:
!pip install uv
!uv sync

## 4. Set up vendored Demucs

Demucs is vendored at `vendor/demucs/`. Add it to PYTHONPATH:

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "vendor", "demucs"))

# Verify imports
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

import demucs
print(f"demucs import OK")

## 5. Inference: separate a sample file

Upload your audio file, then run separation:

In [ ]:
from google.colab import files
print("Upload your audio file:")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"Uploaded: {audio_file}")

In [ ]:
# Separate using the pretrained htdemucs as a baseline
# (use your trained model signature if you have one)
from inference.separate import separate
from pathlib import Path

stem_path = separate(
    input_path=Path(audio_file),
    sig="htdemucs",       # or your trained model sig
    stem="chinese-instrument",
    device="cuda",
)
print(f"\nExtracted stem: {stem_path}")

## 6. Listen to the result

In [ ]:
import soundfile as sf
from IPython.display import Audio

if stem_path.exists():
    data, sr = sf.read(str(stem_path))
    print("Extracted instrument stem:")
    Audio(data.T, rate=sr)
else:
    # Demucs may output to a different path
    # Check separated/<sig>/ for the output
    for p in Path("separated").rglob("**/chinese-instrument.wav"):
        data, sr = sf.read(str(p))
        print(f"Found: {p}")
        Audio(data.T, rate=sr)
        break

## 7. Download the separated stem

In [ ]:
!mkdir -p download
!cp {stem_path} download/
!zip -r separated_stems.zip separated/ download/ 2>/dev/null
files.download('separated_stems.zip')

## 8. (Optional) Build a synthetic dataset

If you have your own instrument clips and background music, upload them and build a dataset:

In [ ]:
# Upload your source clips and backgrounds first, then:
!mkdir -p source_clips backgrounds

# Build a small dataset (50 train, 10 valid)
!uv run --env PYTHONPATH=vendor/demucs python data_pipeline/build_dataset.py \
    --source-dir source_clips/ \
    --bg-dir backgrounds/ \
    --source-name erhu \
    --num-train 50 --num-valid 10 \
    --seg-len 8
print("Dataset built at data/instrument_dataset/")

## 9. (Optional) Train on Colab

Training requires the dataset built above. Use the warm-start checkpoint patcher then launch training:

In [ ]:
# Warm-start from htdemucs
!uv run --env PYTHONPATH=vendor/demucs python training/patch_checkpoint.py \
    --sources erhu other --out outputs/warmstart.th

# Launch training (adjust epochs as needed)
!uv run --env PYTHONPATH=vendor/demucs dora run -d \
    model=htdemucs dset=instrument variant=instrument_ft \
    epochs=50

---

**That's it!** You now have an instrument stem separated from your audio. For more details, see the [repo docs](https://github.com/wushanyun64/chinese-instrument-demucs/tree/main/docs).